# 09 - Calibration: score to trust scoreWithout this, 'trust score' is a renamed softmax output and a reviewer will say so. Calibration set is disjoint from both train and test. Report Brier and ECE, and produce the reliability diagram.

In [ ]:
# --- standard header: every notebook starts with exactly this ---from google.colab import drive; drive.mount('/content/drive')REPO = '/content/secure-dns-trust-ai'!git -C {REPO} pull -q 2>/dev/null || git clone -q https://github.com/sandesh20lamichhane/secure-dns-trust-ai.git {REPO}import sys, os; sys.path.insert(0, REPO)os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'%load_ext autoreload%autoreload 2from src.utils import config, manifest, seeds, ioP = config.paths(); config.ensure_tree(P); seeds.set_all(42)print('repo', manifest.git_sha(REPO))

In [ ]:
import pandas as pd, numpy as npfrom src.models.calibrate import Calibrator, reliability_curvefrom src.evaluate import metrics, predictionscfg = config.load('calibration')RUN = 'fusion_family_disjoint_v1_s42_0004'   # run_id to calibratepred = predictions.load(RUN, P['artifacts']['predictions'])

In [ ]:
# Calibrate on the validation predictions, evaluate on testfor method in cfg['methods']:    cal = Calibrator(method).fit(pred_val['raw_score'], pred_val['true_label'])    p = cal.transform(pred['raw_score'])    m = metrics.evaluate(pred['true_label'], p)    print(method, 'ECE', round(m['ece'],4), 'Brier', round(m['brier_score'],4))

In [ ]:
rows = reliability_curve(pred['true_label'], p, n_bins=cfg['ece_bins'])pd.DataFrame(rows).to_csv(f"{P['results']['tables']}/reliability_{RUN}.csv", index=False)

In [ ]:
# Operating points, chosen by cost rather than by conventionfor op in cfg['operating_points']:    if 'target_tpr' in op:        thr = metrics.threshold_at_tpr(pred['true_label'], p, op['target_tpr'])        print(op['name'], 'threshold', round(thr,4),              'FPR', round(metrics.fpr_at_tpr(pred['true_label'], p, op['target_tpr']),5))